In [1]:
import hashlib
import json
import os

import random
from abc import ABC, abstractmethod
from itertools import product

import numpy as nps
import torch
from tqdm import tqdm

from datasetgenerator.configs import AudioDatasetConfig, DatasetConfig, load_configs
from datasetgenerator.pipelines import (
    AudioProcessingPipeline
)
from datasetgenerator.processors import HuggingFaceProcessor
from datasets import Audio, load_dataset

In [ ]:
# configs = load_configs("datasetgenerator/configs/configs/gliclass-audio.json", "audio")
pipelines, configs = load_configs("configs/configs/dataset/test.json", "configs/configs/processor/huggingface_processor.json", "audio")
processor= HuggingFaceProcessor(configs, pipelines=pipelines)
processor.process(out_dataset_name="gliclass-audio-datset")


## Anotator

In [8]:
!jupyter nbextension enable --py widgetsnbextension --sys-prefix

Enabling notebook extension jupyter-js-widgets/extension...
      - Validating: OK


In [2]:
from datasets import load_dataset, Audio

In [1]:
import hashlib
import json
import random
import os

import random
from abc import ABC, abstractmethod
from itertools import product

import numpy as nps
import torch
from tqdm import tqdm

from datasetgenerator.configs import AudioDatasetConfig, DatasetConfig, load_configs
from datasetgenerator.pipelines import (
    AudioProcessingPipeline
)
from datasetgenerator.processors import HuggingFaceProcessor
from datasets import Audio, load_dataset, Dataset

In [2]:
from typing import Callable
from datasetgenerator.pipelines import ProcessingPipeline
from vllm  import LLM, SamplingParams
from vllm.sampling_params import GuidedDecodingParams
from transformers import AutoTokenizer, AutoModelForCausalLM
from pydantic import BaseModel

INFO 05-27 16:12:32 [__init__.py:239] Automatically detected platform cuda.


In [ ]:
SYSTEMMESSAGE = {
    "role": "user",
    "content": (
        "You are an advanced assistant trained to classify input text into relevant categories (labels) in English ONLY. \n"
        "Your task is to generate a JSON object with two fields:\n"
        "- 'true_labels': up to 25 labels that are accurate and contextually appropriate for the input.\n"
        "- 'false_labels': up to 25 incorrect but contextually challenging (hard negative) labels. These must be:\n"
        "   • Semantically or topically close to the true labels,\n, for example, spf-15 as true label and spf-30 as false"
        "   • Plausible but factually or contextually wrong,\n"
        "   • Never completely random, absurd, or trivially incorrect.\n\n if true label is president, false should be like vise-president, not python programming language (not random)"
        "There could be less then 25 labels if the text is short. The idea is that labels should be really related to the specific content"
        "The output must be a VALID JSON object, structured as:\n"
        '{"true_labels": ["..."], "false_labels": ["..."]}'
    ),
}

class OutputJSON(BaseModel):
    true_labels: list[str]
    false_labels: list[str]

class AnnotatorPipeline(ProcessingPipeline):
    def __init__(self, anotator: str = "Qwen/Qwen2.5-14B-Instruct"):
        self.steps: list[tuple[str, Callable, dict]] = []
        self.init_anotator(anotator)
        self._load_default_steps()

    def _load_default_steps(self):
        self.add_step(
            "anotate_dataset",
            self.annotate_dataset
        )

    def init_anotator(self, anotator: str):
        self.tokenizer = AutoTokenizer.from_pretrained(anotator)
        self.llm = LLM(model=anotator, max_model_len = 8192, tensor_parallel_size=1, dtype="half", gpu_memory_utilization = 0.9, quantization = None)
        json_schema = OutputJSON.model_json_schema()
        guided_decoding_params_json = GuidedDecodingParams(json=json_schema)
        self.sampling_params = SamplingParams(temperature= 0.7 , repetition_penalty = 1.1, top_k=100, max_tokens=1024, top_p=0.8, stop="<end>",
                                              guided_decoding=guided_decoding_params_json)

    def get_text_to_label_few_shot_messages(self, text, examples):
        messages = [SYSTEMMESSAGE]

        example = random.choice(examples)
        input_message = {
            "role": "user",
            "content": (
                f'Here is an example input text: "{example["text"]}"\n'
                "Generate realistic true and hard false labels. English ONLY. \n"
                "Output in VALID JSON:\n"
                '{"true_labels": ["..."], "false_labels": ["..."]}'
            ),
        }
        messages.append(input_message)

        example_message = {
            "role": "assistant",
            "content": json.dumps({
                "true_labels": example["true_labels"],
                "false_labels": example["false_labels"]
            }, ensure_ascii=False)
        }
        messages.append(example_message)

        input_message = {
            "role": "user",
            "content": (
                f'Here is an input text: "{text}"\n'
                "Generate up to 25 true labels and up to 25 hard false labels. English ONLY. \n"
                "False labels must be related but factually or contextually incorrect (hard negatives).\n"
                "Output a VALID JSON:\n"
                '{"true_labels": ["..."], "false_labels": ["..."]}'
            ),
        }
        messages.append(input_message)

        return messages

    def annotate_batch(self, batch_chats: list[str]) -> list[dict]:
        outputs = self.llm.generate(batch_chats, sampling_params=self.sampling_params)
        batch_results = [output.outputs[0].text for output in outputs]
        return batch_results

    def parse_batch_results(self, batch_results: list[str], batch_idxs: list[int]) -> list[dict]:
        parsed_results = []
        for id_, result in zip(batch_idxs, batch_results):
            try:
                parsed = json.loads(result)
                parsed_results.append((id_, parsed))
            except json.JSONDecodeError as e:
                print(f"Failed to decode JSON result [{id_}]: {e}")
            except Exception as e:
                print(f"Failed to parse result [{id_}]: {e}")
        return parsed_results

    def annotate_dataset(self, dataset: Dataset, config: AudioDatasetConfig) -> str:
        batch_size = 2
        annotation_column = dataset["asr_transcript"]
        example_dataset = json.load(open('./datasets/example/example_dataset.json', 'r', encoding='utf-8'))
        
        batch_idxs = []
        batch_chats = []

        parsed_results = []
        for idx, text in enumerate(tqdm(annotation_column, desc="Annotating dataset")):
            messages = self.get_text_to_label_few_shot_messages(text, example_dataset)
            chat =  self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            batch_idxs.append(idx)
            batch_chats.append(chat)
            if len(batch_chats) == batch_size:
                try:
                    batch_results = self.annotate_batch(batch_chats)
                except Exception as err:
                    print(err)
                    continue

                parsed_results.extend(self.parse_batch_results(batch_results, batch_idxs))
                batch_idxs = []
                batch_chats = []

        if batch_chats:
            print("Called last batch with size:", len(batch_chats))
            try:
                batch_results = self.annotate_batch(batch_chats)
            except Exception as err:
                print(err)

            parsed_results.extend(self.parse_batch_results(batch_results, batch_idxs))
        
        dataset = self.update_dataset_with_annotations(dataset, parsed_results, config)
        return dataset

    def update_dataset_with_annotations(self, dataset: Dataset, parsed_results: list[tuple[int, dict]], config: AudioDatasetConfig):
        # Create dictionaries to hold the new values
        all_labels_dict = {idx: result["true_labels"] + result["false_labels"] 
                        for idx, result in parsed_results}
        true_labels_dict = {idx: result["true_labels"] 
                        for idx, result in parsed_results}
        
        def update_example(example, idx):
            if idx in all_labels_dict:
                example["all_labels"] = all_labels_dict[idx]
                example["label"] = true_labels_dict[idx]  # rewrite with config
            else:
                example["all_labels"] = []
                example["label"] = []
            return example
        
        return dataset.map(
            update_example, 
            with_indices=True,
            desc="Updating dataset with annotations"
        )





In [1]:
from datasets import Audio, load_dataset, Dataset

In [2]:
dataset = load_dataset("disco-eth/EuroSpeech", "uk", split="train")
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))

Resolving data files:   0%|          | 0/156 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/40 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/37 [00:00<?, ?it/s]

In [ ]:
dataset["asr_transcript"]

Dataset({
    features: ['audio', 'key', 'country', 'language', 'video_id', 'transcript_id', 'start_seconds', 'end_seconds', 'duration_seconds', 'asr_transcript', 'human_transcript', 'cer', 'wer', 'original_transcript_start_idx', 'original_transcript_end_idx'],
    num_rows: 40791
})

In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM

In [6]:
tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-small")

token_counts = []

for text in dataset["asr_transcript"]:
    tokens = tokenizer.encode(text)
    token_counts.append(len(tokens))

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

/home/werent4/DatasetGenerator/.venv/lib/python3.10/site-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


In [7]:
avg_token_count = sum(token_counts) / len(token_counts)
avg_token_count

45.673040621705766

In [5]:
dataset = dataset.select(range(4))

In [6]:
pipeline = AnnotatorPipeline("Qwen/Qwen2.5-7B-Instruct")

WARNING 05-27 16:12:42 [config.py:2972] Casting torch.bfloat16 to torch.float16.
INFO 05-27 16:12:53 [config.py:717] This model supports multiple tasks: {'score', 'generate', 'classify', 'embed', 'reward'}. Defaulting to 'generate'.
INFO 05-27 16:12:53 [config.py:2003] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 05-27 16:12:56 [core.py:58] Initializing a V1 LLM engine (v0.8.5.post1) with config: model='Qwen/Qwen2.5-7B-Instruct', speculative_config=None, tokenizer='Qwen/Qwen2.5-7B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=8192, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='auto', reasoning_backend=None), observabili

Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]


INFO 05-27 16:14:22 [loader.py:458] Loading weights took 84.09 seconds
INFO 05-27 16:14:22 [gpu_model_runner.py:1347] Model loading took 14.2488 GiB and 84.618886 seconds
INFO 05-27 16:14:35 [backends.py:420] Using cache directory: /home/werent4/.cache/vllm/torch_compile_cache/3fc254abf4/rank_0_0 for vLLM's torch.compile
INFO 05-27 16:14:35 [backends.py:430] Dynamo bytecode transform time: 13.29 s
INFO 05-27 16:14:43 [backends.py:118] Directly load the compiled graph(s) for shape None from the cache, took 7.052 s
INFO 05-27 16:14:44 [monitor.py:33] torch.compile takes 13.29 s in total
INFO 05-27 16:14:47 [kv_cache_utils.py:634] GPU KV cache size: 70,880 tokens
INFO 05-27 16:14:47 [kv_cache_utils.py:637] Maximum concurrency for 8,192 tokens per request: 8.65x
INFO 05-27 16:15:21 [gpu_model_runner.py:1686] Graph capturing finished in 34 secs, took 0.48 GiB
INFO 05-27 16:15:21 [core.py:159] init engine (profile, create kv cache, warmup model) took 58.91 seconds
INFO 05-27 16:15:21 [core_c

In [7]:
parsed_results = pipeline.annotate_dataset(dataset, "audio_dataset_config")

Annotating dataset:   0%|          | 0/4 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/2 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Annotating dataset:  50%|█████     | 2/4 [00:17<00:17,  8.60s/it]

Processed prompts:   0%|          | 0/2 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Annotating dataset: 100%|██████████| 4/4 [00:31<00:00,  7.78s/it]

[(0, {'true_labels': ['benefit_cuts', 'wage_stagnation', 'child_poverty', 'parental_unemployment', 'cost_of_living', 'economic_inequality', 'social_welfare_cuts', 'poverty_rates', 'family_financial_strain', 'income_decrease', 'basic_needs', 'government_policy', '社会保障政策', '教育支出', '医疗保健', '住房补贴', '失业率', '经济增长放缓', '通货膨胀', '税收政策', '劳动市场变化', '社会福利制度', '家庭经济压力', '生活成本危机'], 'false_labels': ['科技公司', '智能手机', '云计算', '人工智能', '电子商务', '网络安全', '汽车制造', '房地产开发', '奢侈品行业', '旅游服务', '体育用品', '家电制造', '金融服务', '能源生产', '农业技术', '航空运输', '食品加工', '时尚设计', '广告制作', '建筑设计', '软件开发', '数据分析', '在线游戏', '音乐创作']}), (1, {'true_labels': ['poverty', 'childhood health', 'social inequality', 'public health', 'deprived areas', 'health disparities', "children's rights", 'well-being', 'social welfare', 'economic disparity', 'health outcomes', 'socioeconomic status', 'community health', 'public policy', 'education', 'nutrition', 'mental health', 'adolescent development', 'government programs', 'child protection', 'health care access'

Updating dataset with annotations:   0%|          | 0/4 [00:00<?, ? examples/s]

In [8]:
len(parsed_results)

4

In [9]:
parsed_results[0]

{'audio': {'path': 'uk_uk_0_05112019_41536_54304.wav',
  'array': array([-0.00509644, -0.00286865, -0.00186157, ...,  0.00262451,
          0.0027771 ,  0.00100708], shape=(204288,)),
  'sampling_rate': 16000},
 'key': 'uk_uk_0_05112019_41536_54304',
 'country': 'uk',
 'language': 'en',
 'video_id': 'uk_0_05112019',
 'transcript_id': 'uk_0_05112019_uk_1_05112019',
 'start_seconds': 41.5359992980957,
 'end_seconds': 54.30400085449219,
 'duration_seconds': 12.767999649047852,
 'asr_transcript': "Because what we've seen is after nine years of cuts, benefit cuts, stagnating wages, the increasing numbers of parents unable to meet the basic costs of living. And the knock-on effects of this reality is a rise in child poverty.",
 'human_transcript': 'Minister will follow. After nine years of cuts, benefit cuts and stagnating wages, an increasing number of parents are unable to meet the basic cost of living, and the knock-on effect of that reality is a rise in child poverty.',
 'cer': 0.1790392